------------------
```markdown
# Copyright © 2025 Meysam Goodarzi
This notebook is licensed under CC BY-NC 4.0 with the following amendments:
- Individuals may use, share, and adapt this material for non-commercial purposes with attribution.
- Institutions/Companies must obtain written consent to use this material, except for nonprofits.
- Commercial use is prohibited without permission.  
Contact: analytica@meysam-goodarzi.com
```
------------------------------
❗❗❗ **IMPORTANT**❗❗❗ **Create a copy of this notebook**

In order to work with this Google Colab you need to create a copy of it. Please **DO NOT** provide your answers here. Instead, work on the copy version. To make a copy:

**Click on: File -> save a copy in drive**

Have you successfully created the copy? if yes, there must be a new tab opened in your browser. Now move to the copy and start from there!

----------------------------------------------


# Agent-Based Modeling II: Pattern Analysis & Parameter Sweeps

## Learning Objectives

By the end of this notebook, you will be able to:
1. Apply Pattern-Oriented Modeling (POM) principles to validate ABMs
2. Conduct parameter sweeps and sensitivity analysis
3. Distinguish between verification and validation
4. Analyze model behavior across different parameter settings
5. Interpret emergence and tipping points in complex systems

## Recap: What We Built Last Time

In the previous session, we:
- ✓ Implemented the complete Schelling segregation model
- ✓ Created Agent and Environment classes
- ✓ Built the simulation loop with happiness evaluation and movement
- ✓ Visualized initial and final states
- ✓ Observed emergence: 30% tolerance → ~75% segregation

**This week:** We analyze the model systematically to understand **why** and **when** segregation emerges.

## Setup and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap
from IPython.display import HTML
import seaborn as sns
import pandas as pd
from tqdm.notebook import tqdm
import random

# Set random seed
np.random.seed(42)
random.seed(42)

# Configuration
GRID_SIZE = 50
AGENT_TYPE_A = 1
AGENT_TYPE_B = 2
EMPTY_CELL = 0
NEIGHBORHOOD_RADIUS = 1
DEFAULT_SIMILARITY_THRESHOLD = 0.3

COLOR_EMPTY = '#FFFFFF'
COLOR_TYPE_A = '#FF6B6B'
COLOR_TYPE_B = '#4ECDC4'
COLOR_UNHAPPY = '#FFD93D'

In [ ]:
# Agent class from Session 1
class Agent:
    """Agent with position, type, and happiness evaluation."""

    def __init__(self, x, y, agent_type, similarity_threshold=DEFAULT_SIMILARITY_THRESHOLD, neighborhood_radius=NEIGHBORHOOD_RADIUS):
        self.x = x
        self.y = y
        self.agent_type = agent_type
        self.similarity_threshold = similarity_threshold
        self.neighborhood_radius = neighborhood_radius

    def get_neighbor_positions(self):
        """Get positions of neighboring cells."""
        positions = []
        for dx in range(-self.neighborhood_radius, self.neighborhood_radius + 1):
            for dy in range(-self.neighborhood_radius, self.neighborhood_radius + 1):
                if dx == 0 and dy == 0:
                    continue
                nx = (self.x + dx) % GRID_SIZE
                ny = (self.y + dy) % GRID_SIZE
                positions.append((nx, ny))
        return positions

    def is_happy(self, env):
        """
        Check if agent is satisfied with current neighborhood composition.

        An agent is happy if the fraction of similar neighbors meets or exceeds
        their similarity threshold. Empty cells are ignored in the calculation.

        Args:
            env (Environment): The environment containing the grid.

        Returns:
            bool: True if agent is happy, False otherwise.
        """
        neighbor_positions = self.get_neighbor_positions()
        similar_count = 0
        total_count = 0

        for nx, ny in neighbor_positions:
            neighbor_type = env.grid[nx, ny]
            total_count += 1
            if neighbor_type == EMPTY_CELL:
                total_count -= 1
            if neighbor_type == self.agent_type:
                similar_count += 1


        if total_count == 0:
            return False

        similarity = similar_count / total_count
        return similarity >= self.similarity_threshold

In [ ]:
# Environment class from Session 1
class Environment:
    """Grid environment with agent placement and movement."""

    def __init__(self):
        self.grid = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)

    def get_empty_cells(self):
        """Get positions of empty cells."""
        empty_cells = []
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                if self.grid[x, y] == EMPTY_CELL:
                    empty_cells.append((x, y))
        return empty_cells

    def place_agent(self, agent, x, y):
        """Place an agent on the grid.

        Args:
            agent (Agent): The agent to be placed.
            x (int): The x-coordinate.
        """
        if self.grid[x, y] != EMPTY_CELL:
            raise ValueError(f"Cell ({x}, {y}) is already occupied")
        self.grid[x, y] = agent.agent_type
        agent.x = x
        agent.y = y

    def remove_agent(self, agent):
        """Remove an agent from the grid."""
        self.grid[agent.x, agent.y] = EMPTY_CELL

    def move_agent(self, agent, new_x, new_y):
        """Move an agent to a new position.

        Args:
            agent (Agent): The agent to be moved.
        """
        if self.grid[new_x, new_y] != EMPTY_CELL:
            raise ValueError(f"Cannot move to occupied cell ({new_x}, {new_y})")
        self.grid[agent.x, agent.y] = EMPTY_CELL
        self.grid[new_x, new_y] = agent.agent_type
        agent.x = new_x
        agent.y = new_y

    def get_occupancy_rate(self):
        """Calculate the occupancy rate of the grid."""
        occupied = np.sum(self.grid != EMPTY_CELL)
        total = GRID_SIZE * GRID_SIZE
        return occupied / total

In [ ]:
# Metrics functions
def calculate_happiness_rate(agents, env):
    """Calculate fraction of happy agents."""
    happy = sum(1 for agent in agents if agent.is_happy(env))
    return happy / len(agents) if agents else 0

def calculate_segregation_index(grid):
    """Calculate average neighbor similarity."""
    similarities = []
    for x in range(GRID_SIZE):
        for y in range(GRID_SIZE):
            if grid[x, y] == EMPTY_CELL:
                continue
            agent_type = grid[x, y]
            similar, total = 0, 0
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    if dx == 0 and dy == 0:
                        continue
                    nx, ny = (x + dx) % GRID_SIZE, (y + dy) % GRID_SIZE
                    if grid[nx, ny] != EMPTY_CELL:
                        total += 1
                        if grid[nx, ny] == agent_type:
                            similar += 1
            if total > 0:
                similarities.append(similar / total)
    return np.mean(similarities) if similarities else 0

# Simulation class
class SchellingModel:
    """Complete Schelling model."""

    def __init__(self, n_agents=2000, occupancy=0.8, similarity_threshold=DEFAULT_SIMILARITY_THRESHOLD):
        self.n_agents = min(n_agents, int(GRID_SIZE * GRID_SIZE * occupancy))
        self.similarity_threshold = similarity_threshold
        self.env = Environment()
        self.agents = []
        self.timestep = 0
        self.history = {'happiness': [], 'segregation': []}
        self._initialize_agents()

    def _initialize_agents(self):
        positions = [(x, y) for x in range(GRID_SIZE) for y in range(GRID_SIZE)]
        random.shuffle(positions)
        for i in range(self.n_agents):
            x, y = positions[i]
            agent_type = AGENT_TYPE_A if i < self.n_agents // 2 else AGENT_TYPE_B
            agent = Agent(x, y, agent_type, self.similarity_threshold)
            self.env.place_agent(agent, x, y)
            self.agents.append(agent)

    def step(self):
        random.shuffle(self.agents)
        moves = 0
        for agent in self.agents:
            if not agent.is_happy(self.env):
                empty = self.env.get_empty_cells()
                if empty:
                    new_x, new_y = random.choice(empty)
                    self.env.move_agent(agent, new_x, new_y)
                    moves += 1
        self.timestep += 1
        self.history['happiness'].append(calculate_happiness_rate(self.agents, self.env))
        self.history['segregation'].append(calculate_segregation_index(self.env.grid))
        return moves

    def run(self, max_steps=100, verbose=False):
        for step in range(max_steps):
            moves = self.step()
            if verbose and step % 10 == 0:
                print(f"Step {step}: {moves} moves, "
                      f"Happiness={self.history['happiness'][-1]:.2%}, "
                      f"Segregation={self.history['segregation'][-1]:.2%}")
            if moves == 0:
                if verbose:
                    print(f"Converged at step {step}")
                return step
        if verbose:
            print(f"Reached max steps ({max_steps})")
        return max_steps

print("Model classes loaded successfully!")

## Pattern-Oriented Modeling (POM)
**Pattern-Oriented Modeling** is the scientific method applied to ABM development:
- Use multiple patterns from real systems to guide model design
- Patterns constrain both structure (what to include) and parameters (how to set values)
- Matching multiple patterns simultaneously reduces uncertainty

### POM Workflow

1. **Identify characteristic patterns** from real system
2. **Build initial model** (minimal version)
3. **Test pattern reproduction**
4. **Diagnose mismatches**
5. **Revise model** structure or parameters
6. **Iterate** until satisfactory matching

### Characteristic Patterns in Urban Segregation

Real-world residential segregation exhibits these patterns:

1. **High segregation levels**: 70-90% same-group neighbors in segregated cities
2. **Spatial clustering**: Contiguous neighborhoods (not random scattering)
3. **Persistence**: Once established, segregation is stable
4. **Mild preferences**: Survey data shows moderate tolerance (not extreme)
5. **Tipping points**: Rapid segregation after threshold change
6. **Scale dependence**: Segregation varies by geographic scale

**Model validation:** Does our Schelling model reproduce these patterns?

### Test Pattern Reproduction

Let's check if our model matches Pattern #1 (high segregation from mild preferences):

In [ ]:
# Run model with mild preference (30%)
print("Testing Pattern #1: Mild preferences → High segregation")
print("="*60)

model = SchellingModel(n_agents=2000, occupancy=0.8, similarity_threshold=0.3)
print(f"Initial setup: {len(model.agents)} agents, tolerance = 30%")
print(f"Expected from pattern: Final segregation = 70-90%\n")

model.run(max_steps=100, verbose=False)

final_seg = model.history['segregation'][-1]
print(f"\nModel result: Final segregation = {final_seg:.1%}")

if 0.70 <= final_seg <= 0.90:
    print("✓ Pattern MATCHED: Model reproduces observed segregation levels")
else:
    print("✗ Pattern NOT matched: Model needs adjustment")

## Verification vs. Validation

The words verfication and validaton are often deployed interchangeably. However, in the realm of ABM, they have different meaning. The following table describes the differences.

| Verification | Validation |
|--------------|------------|
| Are we building the model **right**? | Are we building the **right** model? |
| Is the code bug-free? | Does model represent reality? |
| Implementation matches design? | Outputs match real data? |
| **Technical correctness** | **Scientific credibility** |

**Both are necessary!** A correctly coded wrong model is useless. A great model with bugs is also useless.

### Verification Tests

Let's verify our implementation with boundary condition tests:
1. What happens if the tolerance is set to zero?

In [ ]:
print("VERIFICATION TESTS")
print("="*60)

# Test 1: Tolerance = 0 (everyone unhappy)
print("\nTest 1: Tolerance = 0% (everyone should be happy initially)")
model_t0 = SchellingModel(n_agents=1000, occupancy=0.8, similarity_threshold=0.0)
initial_happiness = calculate_happiness_rate(model_t0.agents, model_t0.env)
print(f"Initial happiness: {initial_happiness:.1%}")
print(f"Expected: 100% (all agents want 0% similar neighbors, meaning that they do not care about who their neibor is)")
if initial_happiness > 0.95:
    print("✓ Test PASSED")
else:
    print("✗ Test FAILED")

2. What if the tolerance is set to 100%?

In [ ]:
# Test 2: Tolerance = 100% (impossible to satisfy)
print("\nTest 2: Tolerance = 100% (want all similar neighbors)")
model_t100 = SchellingModel(n_agents=1000, occupancy=0.8, similarity_threshold=1.0)
initial_happiness = calculate_happiness_rate(model_t100.agents, model_t100.env)
print(f"Initial happiness: {initial_happiness:.1%}")
print(f"Expected: ~0% (very unlikely to have all similar neighbors randomly)")
if initial_happiness < 0.20:
    print("✓ Test PASSED")
else:
    print("✗ Test FAILED")

3. Is the number of agents at the beginning equal the end of the simulation?

In [ ]:
# Test 3: Agent count conservation
print("\nTest 3: Agent count conservation during simulation")
model_test = SchellingModel(n_agents=500, occupancy=0.5, similarity_threshold=0.3)
initial_count = len(model_test.agents)
model_test.run(max_steps=20, verbose=False)
final_count = len(model_test.agents)
print(f"Initial agents: {initial_count}")
print(f"Final agents: {final_count}")
if initial_count == final_count:
    print("✓ Test PASSED: Agent count preserved")
else:
    print("✗ Test FAILED: Agents were created or destroyed!")

4. Is the rate of segregation constant during the simulation or does it increase?

In [ ]:
# Test 4: Segregation should increase
print("\nTest 4: Segregation should increase (or stay same)")
model_seg = SchellingModel(n_agents=1000, occupancy=0.8, similarity_threshold=0.3)
initial_seg = calculate_segregation_index(model_seg.env.grid)
model_seg.run(max_steps=50, verbose=False)
final_seg = calculate_segregation_index(model_seg.env.grid)
print(f"Initial segregation: {initial_seg:.2%}")
print(f"Final segregation: {final_seg:.2%}")
if final_seg >= initial_seg:
    print("✓ Test PASSED: Segregation increased or stayed same")
else:
    print("✗ Test FAILED: Segregation decreased (unexpected!)")

print("\n" + "="*60)
print("Verification complete! All tests should pass.")

## Sensitivity Analysis
To construct a profound understanding of the ABMs we need to see how sensitive the outcomes are when we change the parameter values. This is conducted in two manners:
1. Local sensitivity analysis
2. Global sensitivity analysis

### Local Sensivity Analysis (Parameter Sweeps)

The goal here is to understand how outcomes vary with parameters:
- Identify thresholds/tipping points
- Find parameter ranges that match observations
- Reveal non-linear relationships (emergence!)

We can take the following steps to perform parameter sweeps:

1. Choose parameter to vary (e.g., tolerance: 0.1 to 0.7)
2. Run model for each value (multiple replicates for stochastic models)
3. Record outcome metrics
4. Plot relationships

The above procedure is often called **local sensitivity analysis**.

In [ ]:
# Parameter sweep: Tolerance threshold
print("PARAMETER SWEEP: Tolerance Threshold")
print("="*60)

tolerance_values = np.arange(0.1, 1, 0.1)
n_replicates = 5  # Multiple runs per parameter value

results = []

for tolerance in tqdm(tolerance_values, desc="Tolerance sweep"):
    for rep in range(n_replicates):
        # Run model
        model = SchellingModel(
            n_agents=1500,
            occupancy=0.8,
            similarity_threshold=tolerance
        )
        steps = model.run(max_steps=100, verbose=False)

        # Record results
        results.append({
            'tolerance': tolerance,
            'replicate': rep,
            'final_segregation': model.history['segregation'][-1],
            'final_happiness': model.history['happiness'][-1],
            'convergence_time': steps
        })

# Convert to DataFrame
df_results = pd.DataFrame(results)

print("\nSweep complete!")
print(f"Total runs: {len(results)}")
print("\nSample results:")
print(df_results.head(20))

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Segregation vs Tolerance
ax1 = axes[0]
# Plot individual runs
for rep in range(n_replicates):
    data_rep = df_results[df_results['replicate'] == rep]
    ax1.plot(data_rep['tolerance'], data_rep['final_segregation'],
             'o-', alpha=0.3, color='steelblue')

# Plot mean with error bars
mean_seg = df_results.groupby('tolerance')['final_segregation'].mean()
std_seg = df_results.groupby('tolerance')['final_segregation'].std()
ax1.errorbar(mean_seg.index, mean_seg.values, yerr=std_seg.values,
             fmt='o-', color='darkblue', linewidth=2, markersize=8,
             capsize=5, label='Mean ± SD')

# Add identity line (if no emergence)
ax1.plot(tolerance_values, tolerance_values, 'r--',
         linewidth=2, alpha=0.5, label='y=x (no emergence)')

ax1.set_xlabel('Tolerance Threshold', fontsize=12)
ax1.set_ylabel('Final Segregation Index', fontsize=12)
ax1.set_title('Segregation vs Tolerance', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot 2: Happiness vs Tolerance
ax2 = axes[1]
mean_hap = df_results.groupby('tolerance')['final_happiness'].mean()
std_hap = df_results.groupby('tolerance')['final_happiness'].std()
ax2.errorbar(mean_hap.index, mean_hap.values, yerr=std_hap.values,
             fmt='o-', color='darkgreen', linewidth=2, markersize=8,
             capsize=5, label='Mean ± SD')
ax2.set_xlabel('Tolerance Threshold', fontsize=12)
ax2.set_ylabel('Final Happiness Rate', fontsize=12)
ax2.set_title('Happiness vs Tolerance', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0.9, 1.01])
ax2.legend()

# Plot 3: Convergence Time vs Tolerance
ax3 = axes[2]
mean_time = df_results.groupby('tolerance')['convergence_time'].mean()
std_time = df_results.groupby('tolerance')['convergence_time'].std()
ax3.errorbar(mean_time.index, mean_time.values, yerr=std_time.values,
             fmt='o-', color='darkorange', linewidth=2, markersize=8,
             capsize=5, label='Mean ± SD')
ax3.set_xlabel('Tolerance Threshold', fontsize=12)
ax3.set_ylabel('Steps to Convergence', fontsize=12)
ax3.set_title('Convergence Time vs Tolerance', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend()

plt.tight_layout()
plt.show()

print("\nKey Findings:")
print("1. Segregation >> Tolerance (EMERGENCE!)")
print("2. Even with tolerance=0.3, segregation reaches ~0.75")
print("3. All agents become happy (happiness → 100%)")
print("4. Higher tolerance → faster convergence")

#### Interpretation: Emergence!

**Key observation:** Segregation ≫ Tolerance

- Tolerance = 30% → Segregation = 75%
- This is **not** a linear relationship!
- The red dashed line (y=x) shows what would happen without emergence
- The blue line shows actual model behavior

**Why does this happen?**
1. Initial random placement → some agents unhappy
2. Unhappy agents move to better locations
3. This creates MORE segregated neighborhoods
4. New segregated neighborhoods make MORE agents unhappy in mixed areas
5. **Positive feedback loop** → segregation amplifies

This demonstrates **strong emergence**: system-level patterns not predictable from individual rules.

### Global Sensitivity Analysis

In addition to parameter sweeps, varying multiple parameters at the same time (global sensitivity) would further deepen our understanding of the ABMs. Let's test **density** (grid occupancy):

In [ ]:
# Two-parameter sweep: Tolerance × Density
print("TWO-PARAMETER SENSITIVITY ANALYSIS")
print("="*60)

tolerance_vals = [0.2, 0.3, 0.4, 0.5]
density_vals = [0.5, 0.7, 0.9]
n_reps = 3

results_2d = []

for tol in tqdm(tolerance_vals, desc="Tolerance"):
    for dens in density_vals:
        for rep in range(n_reps):
            n_agents = int(GRID_SIZE * GRID_SIZE * dens)
            model = SchellingModel(
                n_agents=n_agents,
                occupancy=dens,
                similarity_threshold=tol
            )
            steps = model.run(max_steps=100, verbose=False)

            results_2d.append({
                'tolerance': tol,
                'density': dens,
                'replicate': rep,
                'final_segregation': model.history['segregation'][-1],
                'convergence_time': steps
            })

df_2d = pd.DataFrame(results_2d)
print("\nTwo-parameter sweep complete!")

In [ ]:
# Create heatmap
pivot_table = df_2d.groupby(['tolerance', 'density'])['final_segregation'].mean().reset_index()
pivot_matrix = pivot_table.pivot(index='density', columns='tolerance', values='final_segregation')

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot_matrix.values, cmap='RdYlBu_r', aspect='auto', vmin=0.4, vmax=0.9)

# Set ticks
ax.set_xticks(range(len(tolerance_vals)))
ax.set_xticklabels([f"{t:.1f}" for t in tolerance_vals])
ax.set_yticks(range(len(density_vals)))
ax.set_yticklabels([f"{d:.1f}" for d in density_vals])

# Labels
ax.set_xlabel('Tolerance Threshold', fontsize=13)
ax.set_ylabel('Grid Density', fontsize=13)
ax.set_title('Final Segregation: Two-Parameter Heatmap', fontsize=14, fontweight='bold')

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Segregation Index', fontsize=12)

# Add values as text
for i in range(len(density_vals)):
    for j in range(len(tolerance_vals)):
        text = ax.text(j, i, f"{pivot_matrix.values[i, j]:.2f}",
                      ha="center", va="center", color="black", fontsize=10)

plt.tight_layout()
plt.show()

print("\nInsights:")
print("1. Higher density → MORE segregation (less room to move)")
print("2. Lower tolerance → MORE segregation (more demanding agents)")
print("3. Interaction effects visible: impact depends on both parameters")

## Monte Carlo Replication
Monte Carlo simulation is refered to the process of repeating the simulation of an stochastic model to understand what could happen under different realization of stochastic parameters.

Our model has the following **stochastic elements**:
- Random initial placement
- Random move order
- Random destination selection

**Problem**: Different random seeds → different outcomes

**Solution**: Run many times and analyze **distribution** of outcomes

In [ ]:
# Monte Carlo: 50 runs with same parameters
print("MONTE CARLO REPLICATION ANALYSIS")
print("="*60)

n_runs = 50
mc_results = []

for run in tqdm(range(n_runs), desc="MC runs"):
    model = SchellingModel(
        n_agents=1500,
        occupancy=0.75,
        similarity_threshold=0.3
    )
    steps = model.run(max_steps=100, verbose=False)

    mc_results.append({
        'run': run,
        'final_segregation': model.history['segregation'][-1],
        'convergence_time': steps
    })

df_mc = pd.DataFrame(mc_results)

print(f"\n{n_runs} replications complete!")
print("\nStatistics:")
print(df_mc[['final_segregation', 'convergence_time']].describe())

In [ ]:
# Visualize distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Segregation distribution
ax1 = axes[0]
ax1.hist(df_mc['final_segregation'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
mean_seg = df_mc['final_segregation'].mean()
ax1.axvline(mean_seg, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_seg:.3f}')
ax1.set_xlabel('Final Segregation', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of Final Segregation (50 runs)', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Convergence time distribution
ax2 = axes[1]
ax2.hist(df_mc['convergence_time'], bins=20, edgecolor='black', alpha=0.7, color='coral')
mean_time = df_mc['convergence_time'].mean()
ax2.axvline(mean_time, color='darkred', linestyle='--', linewidth=2, label=f'Mean = {mean_time:.1f}')
ax2.set_xlabel('Convergence Time (steps)', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Convergence Time (50 runs)', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Findings:")
print(f"1. Final segregation: {mean_seg:.3f} ± {df_mc['final_segregation'].std():.3f}")
print(f"2. Range: [{df_mc['final_segregation'].min():.3f}, {df_mc['final_segregation'].max():.3f}]")
print(f"3. Convergence time: {mean_time:.1f} ± {df_mc['convergence_time'].std():.1f} steps")
print("\n4. Variation is MODERATE → Results are robust despite stochasticity")
print("5. Final segregation is consistently high (~0.74-0.76)")

## Summary & Key Takeaways

### What We Learned

1. **Pattern-Oriented Modeling (POM)**:
   - Use multiple patterns to constrain and validate models
   - Schelling model reproduces key segregation patterns
   - Matching multiple patterns reduces uncertainty

2. **Verification vs. Validation**:
   - Verification = technical correctness (bugs, implementation)
   - Validation = scientific credibility (matches reality)
   - Both are necessary!

3. **Parameter Sweeps**:
   - Reveal non-linear relationships (emergence!)
   - Identify tipping points and thresholds
   - Show which parameters matter most

4. **Sensitivity Analysis**:
   - Multiple parameters interact
   - Density affects segregation outcomes
   - Heatmaps visualize parameter space

5. **Monte Carlo Replication**:
   - Quantify stochastic uncertainty
   - Report mean ± standard deviation
   - Results are robust if variation is small

### The Big Picture

**Emergence is real**: Mild individual preferences (30%) → Strong macro segregation (75%)

This demonstrates that:
- Individual intentions ≠ collective outcomes
- Small changes can have large effects
- Feedback loops create self-reinforcing patterns
- Policy interventions must account for dynamics, not just static preferences

**Congratulations! You have finished Notebook 2! Great Job!**
🤗🙌👍👏💪
<!--
# Copyright © 2025 Meysam Goodarzi
This notebook is licensed under CC BY-NC 4.0 with the following amendments:
- Individuals may use, share, and adapt this material for non-commercial purposes with attribution.
- Institutions/Companies must obtain written consent to use this material, except for nonprofits.
- Commercial use is prohibited without permission.  
Contact: analytica@meysam-goodarzi.com.
-->